In [0]:
from pyspark.sql.functions import col, when, datediff, upper, row_number, desc, try_to_timestamp, coalesce, lit, round, last, expr, to_date, sequence, explode
from pyspark.sql.window import Window

# Criar o banco de dados da camada Silver
spark.sql("CREATE DATABASE IF NOT EXISTS workspace.silver")

DataFrame[]

In [0]:
# --- DIM_CONSUMIDORES ---
window_cons = Window.partitionBy("customer_id").orderBy(desc("timestamp_ingestion"))
spark.table("workspace.bronze.tb_customers") \
    .withColumn("rn", row_number().over(window_cons)).filter("rn == 1") \
    .select(
        col("customer_id").alias("id_consumidor"),
        col("customer_zip_code_prefix").alias("prefixo_cep"),
        upper(col("customer_city")).alias("cidade"), # Regra: Maiúsculas [cite: 35]
        upper(col("customer_state")).alias("estado")
    ).write.format("delta").mode("overwrite").saveAsTable("workspace.silver.dim_consumidores")

# --- DIM_PRODUTOS ---
window_prod = Window.partitionBy("product_id").orderBy(desc("timestamp_ingestion"))
spark.table("workspace.bronze.tb_products") \
    .withColumn("rn", row_number().over(window_prod)).filter("rn == 1") \
    .select(
        col("product_id").alias("id_produto"),
        col("product_category_name").alias("categoria_produto"),
        col("product_name_lenght").alias("tamanho_nome_produto"),
        col("product_description_lenght").alias("tamanho_description_produto"),
        col("product_photos_qty").alias("quantidade_fotos"),
        col("product_weight_g").alias("peso_produto_gramas"),
        col("product_length_cm").alias("comprimento_centimetros"),
        col("product_height_cm").alias("altura_centimetros"),
        col("product_width_cm").alias("largura_centimetros")
    ).write.format("delta").mode("overwrite").saveAsTable("workspace.silver.dim_produtos")

# --- DIM_VENDEDORES ---
window_vend = Window.partitionBy("seller_id").orderBy(desc("timestamp_ingestion"))
spark.table("workspace.bronze.tb_sellers") \
    .withColumn("rn", row_number().over(window_vend)).filter("rn == 1") \
    .select(
        col("seller_id").alias("id_vendedor"),
        col("seller_zip_code_prefix").alias("prefixo_cep"),
        upper(col("seller_city")).alias("cidade"), # Regra: Maiúsculas [cite: 81]
        upper(col("seller_state")).alias("estado")
    ).write.format("delta").mode("overwrite").saveAsTable("workspace.silver.dim_vendedores")

In [0]:
# --- FAT_PEDIDOS ---
spark.table("workspace.bronze.tb_orders") \
    .withColumn("status", 
        when(col("order_status") == "delivered", "entregue").when(col("order_status") == "canceled", "cancelado")
        .when(col("order_status") == "shipped", "enviado").when(col("order_status") == "processing", "processando")
        .when(col("order_status") == "invoiced", "faturado").when(col("order_status") == "unavailable", "indisponível")
        .when(col("order_status") == "created", "criado").when(col("order_status") == "approved", "aprovado")
    ) \
    .withColumn("tempo_entrega_dias", datediff(col("order_delivered_customer_date"), col("order_purchase_timestamp"))) \
    .withColumn("tempo_entrega_estimado_dias", datediff(col("order_estimated_delivery_date"), col("order_purchase_timestamp"))) \
    .withColumn("entrega_no_prazo", 
        when(col("order_status") != "delivered", "Não Entregue")
        .when(col("order_delivered_customer_date") <= col("order_estimated_delivery_date"), "Sim")
        .otherwise("Não")
    ) \
    .select(
        col("order_id").alias("id_pedido"), col("customer_id").alias("id_consumidor"),
        col("status"), col("order_purchase_timestamp").alias("data_pedido"),
        "tempo_entrega_dias", "tempo_entrega_estimado_dias", "entrega_no_prazo"
    ).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.fat_pedidos")

# --- FAT_AVALIACOES_PEDIDOS ---
spark.table("workspace.bronze.tb_order_reviews") \
    .select(
        col("review_id").alias("id_avaliacao"), col("order_id").alias("id_pedido"),
        col("review_score").alias("nota_avaliacao"),
        coalesce(col("review_comment_title"), lit("Sem título")).alias("titulo_avaliacao"), 
        coalesce(col("review_comment_message"), lit("Sem comentário")).alias("comentario_avaliacao"),
        try_to_timestamp(col("review_creation_date")).alias("data_criacao_avaliacao"),
        try_to_timestamp(col("review_answer_timestamp")).alias("data_resposta_avaliacao")
    ).filter("data_criacao_avaliacao <= current_timestamp()") \
    .write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.fat_avaliacoes_pedidos")

# --- FAT_PAGAMENTOS ---
spark.table("workspace.bronze.tb_order_payments") \
    .withColumn("tipo_pagamento",
        when(col("payment_type") == "credit_card", "Cartão de Crédito").when(col("payment_type") == "boleto", "Boleto")
        .when(col("payment_type") == "voucher", "Voucher").when(col("payment_type") == "debit_card", "Cartão de Dezito")
        .otherwise("Não Definido")
    ).select(
        col("order_id").alias("id_pedido"), col("payment_sequential").alias("sequencial_pagamento"),
        "tipo_pagamento", col("payment_installments").alias("parcelas_pagamento"),
        col("payment_value").alias("valor_pagamento_brl")
    ).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.fat_pagamentos_pedidos")

print("✅ Tabelas Fato da Silver criadas com sucesso (Schema Overwritten)!")

✅ Tabelas Fato da Silver criadas com sucesso (Schema Overwritten)!


In [0]:
from pyspark.sql.functions import to_date, col, last, round, explode, sequence
from pyspark.sql.window import Window

# 1. Criar calendário contínuo para cobrir os finais de semana (Regra 9 da Silver) 
# O intervalo foi ajustado para cobrir o período histórico do Olist
df_datas = spark.sql("SELECT explode(sequence(to_date('2016-01-01'), to_date('2018-12-31'), interval 1 day)) as data_referencia")

# 2. Preparar os dados da API vindos da Bronze
df_api = spark.table("workspace.bronze.tb_cotacao_dolar") \
    .withColumn("data_referencia", to_date(col("dataHoraCotacao")))

# 3. Configurar a Window Function para buscar o último valor não nulo (fechamento da sexta-feira) 
window_dolar = Window.orderBy("data_referencia").rowsBetween(Window.unboundedPreceding, Window.currentRow)

# 4. Join e preenchimento de lacunas com last(ignorenulls=True) 
df_datas.join(df_api, "data_referencia", "left") \
    .withColumn("cotacao_final", last("cotacaoCompra", ignorenulls=True).over(window_dolar)) \
    .select(
        col("data_referencia").alias("data"), 
        round(col("cotacao_final"), 2).alias("cotacao_usd") # Arredondamento obrigatório 
    ) \
    .write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.dim_cotacao_dolar")

print("✅ Tabela silver.dim_cotacao_dolar criada com sucesso e lacunas preenchidas!")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✅ Tabela silver.dim_cotacao_dolar criada com sucesso e lacunas preenchidas!


In [0]:
from pyspark.sql.functions import col

# --- FAT_ITENS_PEDIDOS ---
# Origem: tb_order_items [cite: 45]
spark.table("workspace.bronze.tb_order_items") \
    .select(
        col("order_id").alias("id_pedido"),          # order_id -> id_pedido [cite: 46]
        col("order_item_id").alias("id_item"),       # order_item_id -> id_item [cite: 46]
        col("product_id").alias("id_produto"),       # product_id -> id_produto [cite: 46]
        col("seller_id").alias("id_vendedor"),       # seller_id -> id_vendedor [cite: 46]
        col("price").alias("preco_BRL"),             # price -> preco_BRL [cite: 47]
        col("freight_value").alias("preco_frete")    # freight_value -> preco_frete [cite: 48]
    ).write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.fat_itens_pedidos")

print("✅ Tabela silver.fat_itens_pedidos criada!")

✅ Tabela silver.fat_itens_pedidos criada!


In [0]:
from pyspark.sql.functions import col, round, to_date, sum

# 1. Carregar as tabelas da Silver para o Join
df_pedidos = spark.table("workspace.silver.fat_pedidos")
df_pagamentos = spark.table("workspace.silver.fat_pagamentos_pedidos")
df_cotacao = spark.table("workspace.silver.dim_cotacao_dolar")

# 2. Agrupar pagamentos por pedido (um pedido pode ter várias formas de pagamento)
df_pagos_total = df_pagamentos.groupBy("id_pedido").agg(
    sum("valor_pagamento_brl").alias("valor_total_pago_brl")
)

# 3. Join Final: Pedidos + Pagamentos + Cotação
# Usamos to_date para garantir que o join com a cotação seja feito apenas pela data
df_final = df_pedidos.join(df_pagos_total, "id_pedido", "inner") \
    .withColumn("data_referencia_join", to_date(col("data_pedido"))) \
    .join(df_cotacao, col("data_referencia_join") == col("data"), "left")

# 4. Seleção de colunas e arredondamento (Regra: Exatamente 2 casas decimais)
df_fat_pedido_total = df_final.select(
    col("id_pedido"),
    col("id_consumidor"),
    col("status"),
    round(col("valor_total_pago_brl"), 2).alias("valor_total_pago_brl"), # 
    round(col("valor_total_pago_brl") / col("cotacao_usd"), 2).alias("valor_total_pago_usd"), # [cite: 96, 97]
    col("data_pedido")
)

# 5. Gravação da Tabela Final
df_fat_pedido_total.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.silver.fat_pedido_total")

# 6. Otimização Física (Delta Optimize com ZORDER)
# Essencial para indexar por ID e Data conforme solicitado 
spark.sql("OPTIMIZE workspace.silver.fat_pedido_total ZORDER BY (id_pedido, data_pedido)")

print("✅ Camada Silver concluída e otimizada com sucesso!")

✅ Camada Silver concluída e otimizada com sucesso!
